# Makale Yeniden Üretimi: *Interpretation of Neural Networks is Fragile* (Ghorbani, Abid ve Zou, 2019)

**Makale:** A. Ghorbani, A. Abid, J. Zou. *Interpretation of Neural Networks is Fragile.* AAAI 2019. [arXiv:1710.10547](https://arxiv.org/abs/1710.10547) · Orijinal kod (TensorFlow): [amiratag/InterpretationFragility](https://github.com/amiratag/InterpretationFragility)

**Kitap bölümü:** Molnar, *Interpretable Machine Learning*, Bölüm 28 – Saliency Maps (Pixel Attribution).

**Seçenek 1 – Zafiyet/Manipülasyon.** Makalenin iddiası: Bir görüntüye insan gözüyle fark edilmeyen, küçük ve sistematik bir gürültü eklenirse modelin **tahmini aynı kalırken** saliency haritası (piksel önem haritası) tamamen değiştirilebilir. Yani "model neye bakıyor?" sorusuna verilen cevap, modelin kendisi değiştirilmeden bozulabilir.

**Bu defterde yapılanlar**

| Bölüm | Ne yapılıyor | Makaledeki karşılığı |
|---|---|---|
| 1 | Veri: Imagenette (ImageNet'in 10 sınıflık alt kümesi) | ImageNet deneyleri |
| 2 | Saliency yöntemleri: Simple Gradient, Integrated Gradients, SmoothGrad | Feature-importance yöntemleri |
| 3 | **Top-k saldırısı**: ilk k önemli pikselin önemini düşür, etiketi koru | Top-k attack |
| 4 | Tek görüntüde canlı demo | Şekil: önce / sonra haritalar |
| 5 | Toplu deney: saldırı vs. **rastgele işaretli gürültü** | Ana tablo: top-k kesişimi, rank korelasyonu |
| 6 | Integrated Gradients üzerinde aynı saldırı | Yöntemler arası karşılaştırma |
| 7 | Gürültü bütçesi (ε) duyarlılığı | Yeni katkı / hiperparametre analizi |
| 8 | **SmoothGrad savunma olarak işe yarıyor mu?** | Yeni katkı |

**Kapsam notu.** Makale SqueezeNet (ImageNet) ve CIFAR-10 modelleri, ayrıca DeepLIFT ve influence function'ları da inceliyor. Burada ağırlığı Colab'da rahat çalışan **ResNet-18** üzerinde Simple Gradient ve Integrated Gradients'a veriyorum. DeepLIFT, mass-center ve hedefli saldırılar kapsam dışı.

In [ ]:
import copy, functools, os, random, tarfile, urllib.request, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import torchvision
from torchvision import transforms
from PIL import Image

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
COL_ADV, COL_RND, COL_CLEAN = "#d62728", "#7f7f7f", "#1f77b4"

# ---- Deney ayarları (Colab GPU için; CPU'da değerleri küçültün) ----
K = 1000              # top-k: önem sıralamasında ilk k piksel (224x224 = 50176 pikselden)
EPS = 8 / 255         # L_inf gürültü bütçesi
BETA = 10.0           # saldırı gradyanı için ReLU yerine softplus(beta)
N_PER_CLASS = 3       # Simple Gradient deneyi: sınıf başına görüntü (toplam 30)
ITERS = 100           # saldırı iterasyonu
IG_STEPS = 10         # Integrated Gradients adım sayısı (makale daha fazla kullanır)
N_PER_CLASS_IG = 1    # IG deneyi: sınıf başına görüntü (toplam 10)
ITERS_IG = 50
print("cihaz:", DEVICE, "| torch", torch.__version__)
if DEVICE == "cpu":
    print("UYARI: GPU yok. Çalışma -> Çalışma zamanı türünü değiştir -> GPU seçin; yoksa deneyler çok uzun sürer.")

## 1. Veri: Imagenette

[Imagenette](https://github.com/fastai/imagenette) (fast.ai), ImageNet'ten alınmış 10 kolay ayırt edilen sınıftan oluşan alt kümedir (tench, English springer, cassette player, chain saw, church, French horn, garbage truck, gas pump, golf ball, parachute). Makale ImageNet kullandığı için doğrudan aynı veri türüdür. Ön eğitimli ImageNet modeli bu sınıfları zaten tanıdığı için ayrıca eğitim gerekmez. Doğrulama kümesi (3925 görüntü) kullanılır, tam `imagenette2-320` ~325 MB'tır.

Defter ilk çalıştığında `veri.csv` manifestini üretir (yol, sınıf, ImageNet indeksi). Bu dosyayı repoya ekleyin; görüntülerin kendisini repoya koymayın, defter indirir.

In [ ]:
URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"
ARCHIVE, ROOT = "imagenette2-320.tgz", "imagenette2-320"

if not os.path.exists(ROOT):
    print("Imagenette indiriliyor (~325 MB)...")
    urllib.request.urlretrieve(URL, ARCHIVE)
    with tarfile.open(ARCHIVE) as t:
        try:
            t.extractall(".", filter="data")
        except TypeError:
            t.extractall(".")

# Imagenette sınıfı -> ImageNet-1k sınıf indeksi (ön eğitimli modelin çıktısıyla eşleşir)
SINIF = {
    "n01440764": ("tench", 0),
    "n02102040": ("English springer", 217),
    "n02979186": ("cassette player", 482),
    "n03000684": ("chain saw", 491),
    "n03028079": ("church", 497),
    "n03394916": ("French horn", 566),
    "n03417042": ("garbage truck", 569),
    "n03425413": ("gas pump", 571),
    "n03445777": ("golf ball", 574),
    "n03888257": ("parachute", 701),
}

rows = []
for wnid in sorted(os.listdir(f"{ROOT}/val")):
    if wnid not in SINIF:
        continue
    d = f"{ROOT}/val/{wnid}"
    for fn in sorted(os.listdir(d)):
        if fn.lower().endswith((".jpeg", ".jpg", ".png")):
            rows.append(dict(path=f"{d}/{fn}", wnid=wnid, sinif=SINIF[wnid][0],
                             imagenet_idx=SINIF[wnid][1]))
veri = pd.DataFrame(rows)
veri.to_csv("veri.csv", index=False)      # veri manifesti: repoya bunu da koyun
print(veri.shape)
print(veri.sinif.value_counts().to_string())

prep = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor()])

def load_img(path):
    return prep(Image.open(path).convert("RGB"))     # (3,224,224), [0,1]

def load_batch(paths):
    return torch.stack([load_img(p) for p in paths])

class Normalized(nn.Module):
    """[0,1] aralığındaki görüntüyü alır, ImageNet normalizasyonunu model içinde uygular.
    Böylece saldırı ve saliency hesapları piksel uzayında ([0,1]) yapılır."""
    def __init__(self, m):
        super().__init__()
        self.m = m
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return self.m((x - self.mean) / self.std)

## 2. Model ve saliency yöntemleri

Model: ImageNet'te ön eğitimli **ResNet-18**. Girdi [0,1] piksel uzayındadır (normalizasyon model içinde), böylece gürültü bütçesi doğrudan piksel cinsinden ifade edilir.

Yöntemler (hepsi, tahmin edilen sınıfın **logit**'ine göre):

* **Simple Gradient:** $S(x)_{ij} = \sum_c \left|\partial f_y(x) / \partial x_{c,i,j}\right|$ (kanallar üzerinden toplanır)
* **Integrated Gradients:** $IG(x) = (x - x')\odot \frac{1}{m}\sum_{s=1}^{m} \partial f_y\!\left(x' + \tfrac{s}{m}(x-x')\right)/\partial x$, taban görüntü $x'$ = siyah görüntü. Piksel önemi = kanallar üzerinde mutlak değer toplamı.
* **SmoothGrad:** gürültülü kopyaların gradyan büyüklüklerinin ortalaması (bölüm 8'de savunma olarak kullanılacak).

Sıralama metrikleri (iki harita arasında): **top-k kesişimi** (ilk k pikselin ortak oranı), **Spearman rank korelasyonu**, **ağırlık merkezi kayması** (piksel).

In [ ]:
def build_model():
    net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    m = Normalized(net).eval().to(DEVICE)
    for mod in m.modules():
        if isinstance(mod, nn.ReLU):
            mod.inplace = False
    for p in m.parameters():
        p.requires_grad_(False)
    return m

def to_softplus(model, beta=BETA):
    """ReLU'nun 2. türevi 0'dır; saldırı gradyanı için ReLU -> Softplus(beta) kopyası."""
    m = copy.deepcopy(model)
    def rep(mod):
        for name, ch in mod.named_children():
            if isinstance(ch, nn.ReLU):
                setattr(mod, name, nn.Softplus(beta=beta))
            else:
                rep(ch)
    rep(m)
    return m

model = build_model()
model_soft = to_softplus(model)

# ---------------- saliency yöntemleri: sal_fn(model, x, label, create_graph) -> (H,W) ----------------
def saliency_sg(model, x, label, create_graph=False):
    if not create_graph:
        x = x.detach().clone().requires_grad_(True)
    logit = model(x)[0, label]
    g, = torch.autograd.grad(logit, x, create_graph=create_graph)
    return g.abs().sum(1)[0]

def integrated_gradients(model, x, label, steps=IG_STEPS, create_graph=False):
    base = torch.zeros_like(x)
    if not create_graph:
        x = x.detach()
    alphas = torch.linspace(1 / steps, 1, steps, device=x.device).view(-1, 1, 1, 1)
    xs = base + alphas * (x - base)
    if not create_graph:
        xs = xs.clone().requires_grad_(True)
    logits = model(xs)[:, label].sum()
    g, = torch.autograd.grad(logits, xs, create_graph=create_graph)
    ig = (x - base) * g.mean(0, keepdim=True)
    return ig.abs().sum(1)[0]

def smoothgrad(model, x, label, n=25, sigma=0.15, seed=0):
    gen = torch.Generator(device=x.device).manual_seed(seed)
    xs = x.detach() + sigma * torch.randn((n,) + tuple(x.shape[1:]), generator=gen, device=x.device)
    xs.requires_grad_(True)
    g, = torch.autograd.grad(model(xs)[:, label].sum(), xs)
    return g.abs().sum(1).mean(0)

SAL = {"sg": saliency_sg, "ig": functools.partial(integrated_gradients, steps=IG_STEPS)}

# ---------------- metrikler ----------------
def topk_mask(sal, k=K):
    flat = sal.flatten()
    m = torch.zeros_like(flat)
    m[torch.topk(flat, k).indices] = 1
    return m.view_as(sal)

def topk_intersection(s1, s2, k=K):
    return (topk_mask(s1, k) * topk_mask(s2, k)).sum().item() / k

def spearman(s1, s2):
    r1 = s1.flatten().argsort().argsort().float()
    r2 = s2.flatten().argsort().argsort().float()
    r1, r2 = r1 - r1.mean(), r2 - r2.mean()
    return ((r1 * r2).sum() / (r1.norm() * r2.norm() + 1e-12)).item()

def center_of_mass(sal):
    h, w = sal.shape
    p = sal / (sal.sum() + 1e-12)
    ys = torch.arange(h, device=sal.device).float()
    xs = torch.arange(w, device=sal.device).float()
    return torch.stack([(p.sum(1) * ys).sum(), (p.sum(0) * xs).sum()])

def com_shift(s1, s2):
    return (center_of_mass(s1) - center_of_mass(s2)).norm().item()

def vis(sal):
    s = sal.detach().cpu().numpy()
    return np.clip(s / (np.percentile(s, 99) + 1e-12), 0, 1)

def to_img(x):
    return x[0].detach().cpu().permute(1, 2, 0).numpy().clip(0, 1)

## 3. Saldırı: top-k saldırısı

Amaç: $x$'in küçük bir komşuluğunda ($\|\delta\|_\infty \le \varepsilon$) öyle bir $x+\delta$ bulmak ki

1. tahmin edilen etiket **değişmesin**,
2. orijinal saliency haritasındaki **ilk k pikselin** önemi mümkün olduğunca düşsün.

Kayıp: $\;\mathcal{L}(x+\delta)=\dfrac{\sum_{(i,j)\in T_k(x)} S(x+\delta)_{ij}}{\sum_{(i,j)} S(x+\delta)_{ij}}$, burada $T_k(x)$ orijinal görüntünün top-k piksel kümesi. Her iterasyonda $\delta \leftarrow \delta - \alpha\,\mathrm{sign}(\nabla_x \mathcal{L})$, sonra $\varepsilon$-topuna ve $[0,1]$'e projeksiyon (iteratif işaretli gradyan, I-FGSM tarzı). Etiket değişirse durulur, en düşük gerçek kesişimi veren nokta saklanır.

Saldırı gradyanı **softplus** kopyada hesaplanır (ReLU'nun ikinci türevi sıfır olduğundan, bu Dombrowski ve ark. 2019'daki gibi standart bir yaklaşımdır); **başarı ölçümü her zaman orijinal ReLU modelindeki gerçek saliency ile** yapılır. Bu, orijinal kodla birebir aynı olmayabilir.

In [ ]:
def topk_attack(model_relu, model_soft, x, label, sal_fn, k=K, eps=EPS, iters=ITERS):
    step = eps / 8
    s0 = sal_fn(model_relu, x, label).detach()
    mask = topk_mask(s0, k)
    x_adv = x.detach().clone()
    best_x, best_ov = x_adv.clone(), 1.0
    history = [1.0]
    for _ in range(iters):
        xa = x_adv.clone().requires_grad_(True)
        s = sal_fn(model_soft, xa, label, create_graph=True)
        loss = (s * mask).sum() / (s.sum() + 1e-12)
        g, = torch.autograd.grad(loss, xa)
        x_new = x_adv - step * g.sign()
        x_new = torch.min(torch.max(x_new, x - eps), x + eps).clamp(0, 1).detach()
        with torch.no_grad():
            if model_relu(x_new).argmax(1).item() != label:
                break                                   # etiket korunmalı
        x_adv = x_new
        ov = topk_intersection(sal_fn(model_relu, x_adv, label).detach(), s0, k)
        history.append(ov)
        if ov < best_ov:
            best_ov, best_x = ov, x_adv.clone()
    return best_x, s0, history

def run_eval(imgs, method, eps=EPS, iters=ITERS, k=K, keep=False):
    """imgs: (3,H,W) tensör listesi. Her görüntü için saldırı + rastgele gürültü karşılaştırması."""
    fn = SAL[method]
    rows, store = [], []
    for i, x in enumerate(imgs):
        x = x.unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            label = model(x).argmax(1).item()
        x_adv, s0, hist = topk_attack(model, model_soft, x, label, fn, k, eps, iters)
        s_adv = fn(model, x_adv, label).detach()

        gen = torch.Generator(device=DEVICE).manual_seed(SEED + i)
        noise = torch.sign(torch.randn(x.shape, generator=gen, device=DEVICE))
        x_rnd = (x + eps * noise).clamp(0, 1)
        s_rnd = fn(model, x_rnd, label).detach()
        with torch.no_grad():
            lab_adv = model(x_adv).argmax(1).item()
            lab_rnd = model(x_rnd).argmax(1).item()

        rows.append(dict(
            kesisim_saldiri=topk_intersection(s0, s_adv, k), kesisim_rastgele=topk_intersection(s0, s_rnd, k),
            spearman_saldiri=spearman(s0, s_adv), spearman_rastgele=spearman(s0, s_rnd),
            com_saldiri=com_shift(s0, s_adv), com_rastgele=com_shift(s0, s_rnd),
            linf_saldiri=(x_adv - x).abs().max().item() * 255,
            etiket_korundu_saldiri=(lab_adv == label), etiket_korundu_rastgele=(lab_rnd == label),
        ))
        if keep:
            store.append(dict(x=x.cpu(), x_adv=x_adv.cpu(), x_rnd=x_rnd.cpu(), label=label,
                              s0=s0.cpu(), s_adv=s_adv.cpu(), s_rnd=s_rnd.cpu(), hist=hist))
    return pd.DataFrame(rows), store

def pick(n_per_class):
    sub = veri.groupby("wnid").sample(n=n_per_class, random_state=SEED).reset_index(drop=True)
    return sub, [load_img(p) for p in sub.path]

## 4. Model gerçekten doğru mu bakıyor? Hızlı sağlık kontrolü ve tek görüntü demosu

Önce seçilen görüntülerde ön eğitimli modelin doğruluğunu ölçüyorum (Imagenette sınıfları ImageNet indekslerine eşlendi). Sonra Simple Gradient için saldırıyı bir görüntüde görsel olarak gösteriyorum.

In [ ]:
sub, imgs = pick(N_PER_CLASS)
with torch.no_grad():
    preds = [model(x.unsqueeze(0).to(DEVICE)).argmax(1).item() for x in imgs]
acc = np.mean([p == t for p, t in zip(preds, sub.imagenet_idx)])
print(f"{len(imgs)} görüntü | ön eğitimli ResNet-18, Imagenette üzerinde doğruluk: {acc:.2f}")

df_demo, st_demo = run_eval(imgs[:1], "sg", keep=True)
d = st_demo[0]
fig, ax = plt.subplots(1, 5, figsize=(15, 3.2))
ax[0].imshow(to_img(d["x"])); ax[0].set_title("Orijinal")
ax[1].imshow(vis(d["s0"]), cmap="inferno"); ax[1].set_title("Saliency (orijinal)")
ax[2].imshow(to_img(d["x_adv"])); ax[2].set_title(f"Saldırı sonrası (max |δ|={df_demo.linf_saldiri[0]:.1f}/255)")
ax[3].imshow(vis(d["s_adv"]), cmap="inferno")
ax[3].set_title(f"Saliency (saldırı)\ntop-{K} kesişim={df_demo.kesisim_saldiri[0]:.2f}")
ax[4].imshow(vis(d["s_rnd"]), cmap="inferno")
ax[4].set_title(f"Saliency (rastgele gürültü)\ntop-{K} kesişim={df_demo.kesisim_rastgele[0]:.2f}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print("Etiket korundu mu (saldırı / rastgele):", df_demo.etiket_korundu_saldiri[0], "/", df_demo.etiket_korundu_rastgele[0])

## 5. Toplu deney: Simple Gradient

Her görüntü için iki karşılaştırma yapıyorum, ikisi de **aynı** $\varepsilon$ bütçesiyle:

* **Sistematik saldırı** (top-k saldırısı),
* **Rastgele işaretli gürültü** $x + \varepsilon\cdot\mathrm{sign}(\mathcal{N}(0,1))$.

Makalenin ana mesajı: rastgele gürültü haritayı çok az değiştirirken sistematik saldırı kesişimi ciddi biçimde düşürür, üstelik etiket aynı kalır.

In [ ]:
df_sg, st_sg = run_eval(imgs, "sg", keep=True)
ozet_sg = df_sg.drop(columns=["etiket_korundu_saldiri", "etiket_korundu_rastgele"]).agg(["mean", "std"]).T.round(3)
print(f"{len(df_sg)} görüntü | etiket korunma: saldırı={df_sg.etiket_korundu_saldiri.mean():.2f}, rastgele={df_sg.etiket_korundu_rastgele.mean():.2f}")
ozet_sg

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (a, r, ttl) in zip(axes, [("kesisim_saldiri", "kesisim_rastgele", f"Top-{K} kesişimi (1 = aynı harita)"),
                                  ("spearman_saldiri", "spearman_rastgele", "Spearman rank korelasyonu"),
                                  ("com_saldiri", "com_rastgele", "Ağırlık merkezi kayması (piksel)")]):
    ax.boxplot([df_sg[r], df_sg[a]], labels=["rastgele", "saldırı"], patch_artist=True,
               boxprops=dict(facecolor="#eeeeee"))
    ax.set_title(ttl); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(4, 5, figsize=(14, 11))
for r in range(4):
    d = st_sg[r * (len(st_sg) // 4)]
    ov_a = topk_intersection(d["s0"], d["s_adv"]); ov_r = topk_intersection(d["s0"], d["s_rnd"])
    ax[r, 0].imshow(to_img(d["x"])); ax[r, 1].imshow(vis(d["s0"]), cmap="inferno")
    ax[r, 2].imshow(to_img(d["x_adv"])); ax[r, 3].imshow(vis(d["s_adv"]), cmap="inferno")
    ax[r, 4].imshow(vis(d["s_rnd"]), cmap="inferno")
    ax[r, 3].set_title(f"saldırı: kesişim {ov_a:.2f}"); ax[r, 4].set_title(f"rastgele: kesişim {ov_r:.2f}")
    if r == 0:
        ax[r, 0].set_title("Orijinal"); ax[r, 1].set_title("Saliency"); ax[r, 2].set_title("Saldırı sonrası")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

## 6. Integrated Gradients üzerinde aynı saldırı

Makale birden fazla yöntemi karşılaştırıyor. IG hesabı pahalı olduğu için daha az görüntü ve iterasyon kullanıyorum (`N_PER_CLASS_IG`, `ITERS_IG`). IG adım sayısı da makaledekinden küçüktür (`IG_STEPS`); bu sonuçları nitel olarak okuyun.

In [ ]:
sub_ig, imgs_ig = pick(N_PER_CLASS_IG)
df_ig, st_ig = run_eval(imgs_ig, "ig", iters=ITERS_IG, keep=True)
ozet_ig = df_ig.drop(columns=["etiket_korundu_saldiri", "etiket_korundu_rastgele"]).agg(["mean", "std"]).T.round(3)
print(f"{len(df_ig)} görüntü | etiket korunma: saldırı={df_ig.etiket_korundu_saldiri.mean():.2f}")
ozet_ig

## 7. Gürültü bütçesi duyarlılığı ($\varepsilon$)

Saldırı ne kadar küçük bir gürültüyle işe yarıyor? Bir alt küme üzerinde $\varepsilon \in \{2,4,8,16\}/255$ için ortalama top-k kesişimini ölçüyorum.

In [ ]:
eps_list = [2, 4, 8, 16]
imgs_sub = imgs[::3][:10]
rows = []
for e in eps_list:
    d_, _ = run_eval(imgs_sub, "sg", eps=e / 255, iters=60)
    rows.append(dict(eps=e, saldiri=d_.kesisim_saldiri.mean(), rastgele=d_.kesisim_rastgele.mean(),
                     etiket=d_.etiket_korundu_saldiri.mean()))
eps_df = pd.DataFrame(rows).round(3)
display(eps_df) if "display" in globals() else print(eps_df)

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(eps_df.eps, eps_df.saldiri, "o-", color=COL_ADV, label="saldırı")
ax.plot(eps_df.eps, eps_df.rastgele, "s-", color=COL_RND, label="rastgele gürültü")
ax.set(xlabel="ε (0-255 ölçeğinde)", ylabel=f"Top-{K} kesişimi", title="Bütçe arttıkça harita bozuluyor", ylim=(0, 1.02))
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.show()

## 8. Yeni katkı: SmoothGrad savunma olarak işe yarıyor mu?

Saldırı **Simple Gradient**'e karşı optimize edildi. Aynı görüntü çiftini (orijinal, saldırılmış) bu kez **SmoothGrad** ile açıklarsam haritalar hâlâ birbirine benziyor mu? Yüksek kesişim, "gürültülü örnekleme ile ortalama almak saldırıyı zayıflatıyor" demektir.

Dürüstlük notu: saldırı SmoothGrad'e karşı **uyarlanmadı** (adaptive attack yok). Bu yüzden sonuç "SmoothGrad güvenli" demek değil, yalnızca bu saldırının aktarılabilirliğini ölçer.

In [ ]:
rows = []
for d in st_sg:
    x, xa, lab = d["x"].to(DEVICE), d["x_adv"].to(DEVICE), d["label"]
    sg0, sg1 = smoothgrad(model, x, lab), smoothgrad(model, xa, lab)
    rows.append(dict(vanilla_kesisim=topk_intersection(d["s0"].to(DEVICE), d["s_adv"].to(DEVICE)),
                     smoothgrad_kesisim=topk_intersection(sg0, sg1),
                     vanilla_spearman=spearman(d["s0"].to(DEVICE), d["s_adv"].to(DEVICE)),
                     smoothgrad_spearman=spearman(sg0, sg1)))
sm = pd.DataFrame(rows)
print(sm.agg(["mean", "std"]).T.round(3).to_string())

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.bar(["Simple Gradient", "SmoothGrad"], [sm.vanilla_kesisim.mean(), sm.smoothgrad_kesisim.mean()],
       yerr=[sm.vanilla_kesisim.std(), sm.smoothgrad_kesisim.std()], color=[COL_ADV, COL_CLEAN], capsize=4)
ax.set(ylabel=f"Top-{K} kesişimi (orijinal vs saldırılmış)", ylim=(0, 1.05))
ax.spines[["top", "right"]].set_visible(False); plt.show()

## 9. Sonuç özeti (bu çalıştırmanın gerçek sayılarıyla)

In [ ]:
m = df_sg.mean(numeric_only=True)
print(f"""SIMPLE GRADIENT ({len(df_sg)} görüntü, ε={EPS*255:.0f}/255, k={K})
- Etiket korunma: saldırı %{100*df_sg.etiket_korundu_saldiri.mean():.0f} | rastgele %{100*df_sg.etiket_korundu_rastgele.mean():.0f}
- Top-{K} kesişimi: saldırı {m.kesisim_saldiri:.2f} | rastgele {m.kesisim_rastgele:.2f}
- Spearman: saldırı {m.spearman_saldiri:.2f} | rastgele {m.spearman_rastgele:.2f}
- Ağırlık merkezi kayması (px): saldırı {m.com_saldiri:.1f} | rastgele {m.com_rastgele:.1f}
- Gerçek max |δ|: {df_sg.linf_saldiri.max():.1f}/255

INTEGRATED GRADIENTS ({len(df_ig)} görüntü, {IG_STEPS} adım)
- Top-{K} kesişimi: saldırı {df_ig.kesisim_saldiri.mean():.2f} | rastgele {df_ig.kesisim_rastgele.mean():.2f}

SMOOTHGRAD SAVUNMASI
- Top-{K} kesişimi: Simple Gradient {sm.vanilla_kesisim.mean():.2f} -> SmoothGrad {sm.smoothgrad_kesisim.mean():.2f}
""")

## 10. Tartışma ve sınırlar

**Beklenen bulgu (makale):** rastgele gürültü saliency'yi çok az değiştirir, sistematik saldırı ise etiketi koruyarak top-k kesişimini ve rank korelasyonunu belirgin düşürür. Yukarıdaki sayılar bu çalıştırmanın gerçek sonucudur. Sayılar makaledeki değerlerle birebir aynı çıkmak zorunda değil: model (ResNet-18), veri (Imagenette), ε ve iterasyon sayıları farklı.

**Bu kopyanın makaleden farkları**
* Makale SqueezeNet ve CIFAR-10 kullanır, burada ResNet-18.
* Saldırı gradyanı için softplus yaklaşımı kullanıldı; orijinal kodun ikinci türev yaklaşımıyla aynı olmayabilir.
* IG'de adım sayısı ve görüntü sayısı küçük tutuldu.
* DeepLIFT, mass-center ve hedefli saldırılar ile influence function deneyleri yok.
* Gürültü sınırı $L_\infty$; makaledeki ε ile aynı ölçekte olmasına çalıştım ama birebir olduğunu iddia etmiyorum.

**Savunma için olası sorular**
* *Neden işe yarıyor?* Saliency, modelin girdiye göre **gradyanı**dır. Karar sınırı komşuluğunda gradyan hızlı değişebilir; etiket sabit kalsa da gradyan yönü kolayca oynatılabilir (Dombrowski ve ark. 2019'daki "geometri suçlu" argümanı).
* *Neden rastgele gürültü etkisiz?* Rastgele yön, gradyan haritasını bozacak yönle neredeyse dik olur; sistematik saldırı ise haritanın kendi gradyanını (2. türev) kullanır.
* *Nasıl korunulur?* Gradyan ortalaması (SmoothGrad), softplus/yumuşak aktivasyonlu modeller, saliency'nin kendisine karşı eğitim (düzenlileştirme). Hiçbiri kanıtlı garanti vermez.
* *Bu "model yanlış" demek mi?* Hayır. Tahmin değişmiyor; bozulan şey **açıklama**. Bu yüzden saliency tek başına bir denetim aracı olarak güvenilmez.